# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gokul-krishna-rajeev/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

We choose ranking or scoring. We priortize the content items for the editorial team to review. Instead of a 'Yes' or 'No' answer, we seek to rank content items from highest to lowest priority so that the editorial teams can efficiently spend their limited time reviewing pages that need the most attention.

In [18]:
import pandas as pd

# Remote starter CSV dataset
data_path = "https://raw.githubusercontent.com/gokul-krishna-rajeev/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(data_path)
print("Total rows:", len(df))
print("\nContent types in dataset:")
print(df["content_type"].value_counts(dropna=False))

Total rows: 30000

Content types in dataset:
content_type
keyword article       27207
feedly article         2096
comparison article      697
Name: count, dtype: int64


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

We predict a priority score which shows the expected performance decline or recovery potential for the contents over a future time
window.

This is a observed outcome. The label is developed over by the changes in traffic and the amount of impressions in a recurring window
rather creating a static defined threshold.

In [19]:
# Handle target column mapping cleanly from raw dataset
if "is_declining" in df.columns:
    df["is_declining_label"] = df["is_declining"].astype(int)
elif "is_declining_label" not in df.columns:
    # Fallback to trend_direction if raw boolean flag isn't present
    df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

# Inspect target distribution directly
target_related_cols = ["content_id", "clicks_90d", "impressions_90d", "is_declining_label"]
print(df[target_related_cols].head())

print("\nObserved declining label distribution:")
print(df["is_declining_label"].value_counts(normalize=True))

             content_id  clicks_90d  impressions_90d  is_declining_label
0  content_304f48230142          29             3803                   1
1  content_a1fb4e703a9e           7            15320                   1
2  content_9aa793d4d895          11            12581                   1
3  content_331d6c4de07b          58            11751                   0
4  content_d99b7a2d90ca          24            19140                   1

Observed declining label distribution:
is_declining_label
1    0.542067
0    0.457933
Name: proportion, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Metric: Precision@K (specifically Precision@20 or Precision@50) and ROC-AUC.

Achieving a Precision@K higher than the base rate at the target queue capacity (K). This ensures that all the majority top ranked pages flagged for reivew remain as the true candidates for content refresh and maximized edtorial productivity.

In [20]:
# Baseline metric check
base_rate = df["is_declining_label"].mean()
print(f"Base rate of target class: {base_rate:.2%}")
print(f"Random guessing Precision@K baseline: {base_rate:.2%}")


Base rate of target class: 54.21%
Random guessing Precision@K baseline: 54.21%


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = One pseudonymized content item (content_id) for a specific client (client_id)

In [21]:
# Load starter slice and verify unit of analysis
unit_df = df[["content_id", "client_id", "content_type", "clicks_90d", "impressions_90d", "ctr", "avg_position"]].copy()

# Sketch mock ranking score (handling avg_position == 0 gotcha where 0 means no data)
valid_position = unit_df["avg_position"].replace(0, 100)
unit_df["priority_score_sketch"] = (
    (100 - valid_position) * 0.4 +
    unit_df["clicks_90d"] * 0.6
)

print(f"Dataframe Shape: {unit_df.shape}")
print("Check grain uniqueness (should be True):", unit_df.set_index(["content_id", "client_id"]).index.is_unique)
unit_df.head()


Dataframe Shape: (30000, 8)
Check grain uniqueness (should be True): True


,content_id,client_id,content_type,clicks_90d,impressions_90d,ctr,avg_position,priority_score_sketch
0,content_304f48230142,client_f369cb89fc,keyword article,29,3803,0.76,10.6,53.16
1,content_a1fb4e703a9e,client_4e07408562,keyword article,7,15320,0.05,20.3,36.08
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,11,12581,0.09,36.5,32.00
3,content_331d6c4de07b,client_19581e27de,keyword article,58,11751,0.49,6.2,72.32
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,24,19140,0.13,44.0,36.80


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A simple fixed rule falls short due to content performance patterns being tangled and non linear. Factors include  search position, impression rates, traffic interaction across different clients. It also generates false positives or misses out multi signal decay trends, whereas ML rules out these interactions without being held to any static thresholds.

In [22]:
# Measure linear relationship between features and the target label
numeric_cols = ["clicks_90d", "impressions_90d", "ctr", "avg_position", "word_count", "engagement_rate"]
valid_cols = [c for c in numeric_cols if c in df.columns]

correlations = df[valid_cols].apply(lambda col: col.corr(df["is_declining_label"]))
print("Correlation between individual features and decline label:")
print(correlations.round(3))

Correlation between individual features and decline label:
clicks_90d        -0.040
impressions_90d   -0.018
ctr               -0.062
avg_position      -0.029
word_count         0.090
engagement_rate   -0.013
dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.